# Complexity Analysis
This notebook analyzes the profiling statistics to understand the time complexity of different sections of the pipeline.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

# Set plot style
# plt.style.use('seaborn-v0_8')

In [ ]:
# Load the profiling stats
file_path = './profiling_stats_1000.json'

with open(file_path) as f:
    data = json.load(f)

print(f"Loaded {len(data['individual_runs'])} runs.")

In [ ]:
# Process data into DataFrame
runs = []
for run in data['individual_runs']:
    # Flatten the dictionary
    flat_run = {
        'generation': run['generation'],
        'feature_extraction': run['feature_extraction'],
        'simulation': run['simulation'],
        'total': run['total'],
        'qubits': run['details']['qubits'],
        'depth': run['details']['depth'],
    }
    runs.append(flat_run)

df = pd.DataFrame(runs)

# We want to see the accumulation over processed circuits (chronological order of the list)
# So we don't sort. We just use the index as "Number of circuits processed".
df['circuit_index'] = df.index + 1

df.head()

In [ ]:
# Calculate Cumulative Sums for the metrics
df['cumulative_generation'] = df['generation'].cumsum()
df['cumulative_feature_extraction'] = df['feature_extraction'].cumsum()
df['cumulative_simulation'] = df['simulation'].cumsum()

# Plot
plt.figure(figsize=(12, 6))

plt.plot(df['circuit_index'], df['cumulative_generation'], label='Cumulative Generation Time', alpha=0.8)
plt.plot(df['circuit_index'], df['cumulative_feature_extraction'], label='Cumulative Feature Extraction Time', alpha=0.8)
plt.plot(df['circuit_index'], df['cumulative_simulation'], label='Cumulative Simulation Time', alpha=0.8)

plt.title('Cumulative Execution Time vs Number of Circuits Processed')
plt.xlabel('Number of Circuits Processed')
plt.ylabel('Total Time (seconds)')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Logarithmic Scale Plot (Cumulative)
plt.figure(figsize=(12, 6))

plt.plot(df['circuit_index'], df['cumulative_generation'], label='Cumulative Generation Time', alpha=0.8)
plt.plot(df['circuit_index'], df['cumulative_feature_extraction'], label='Cumulative Feature Extraction Time', alpha=0.8)
plt.plot(df['circuit_index'], df['cumulative_simulation'], label='Cumulative Simulation Time', alpha=0.8)

plt.title('Cumulative Execution Time vs Number of Circuits Processed (Log Scale)')
plt.xlabel('Number of Circuits Processed')
plt.ylabel('Total Time (seconds) - Log Scale')
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Linear Scale Plot for Generation and Feature Extraction (excluding Simulation)
plt.figure(figsize=(12, 6))

plt.plot(df['circuit_index'], df['cumulative_generation'], label='Cumulative Generation Time')
plt.plot(df['circuit_index'], df['cumulative_feature_extraction'], label='Cumulative Feature Extraction Time', alpha=0.8)

plt.title('Cumulative Execution Time: Generation vs Feature Extraction',fontsize=18)
plt.xlabel('Number of Circuits Processed',fontsize=18)
plt.ylabel('Total Time (seconds)',fontsize=18)
plt.legend(fontsize=14)
plt.grid(True, which="both", ls="-")
for ax in plt.gcf().get_axes():
    ax.tick_params(axis='both', labelsize=14)
plt.tight_layout()
plt.savefig("cumulative_execution_time_generation_vs_feature_extraction.pdf", format="pdf") 
plt.show()



## Analysis

From the cumulative plot above, we can observe:
1. **Simulation** accumulates time much faster than the other phases, indicating it is the most computationally expensive part of the pipeline.
2. **Generation** and **Feature Extraction** have very slow growth in cumulative time, meaning their per-circuit cost is very low compared to simulation.
3. The slope of the **Simulation** line indicates the average rate of processing; steeper sections might correspond to more complex circuits being processed in that batch.